# 02 - Country dimension

Reads the country mapping CSV from the raw volume and writes it as a Delta table
in the silver schema. This dim table will be joined into the OBT to enrich the
data with full country names and continents.

The mapping was generated with help from an LLM since the dataset uses IOC codes
(Chinese Taipei, RSA for South Africa, etc) which don't match standard ISO 3166.

*Some parts and solutions of this code was found through help from Claude*

In [0]:
import sys
import os

# Make repo root importable so we can use utils/
sys.path.append(os.path.abspath("../.."))

from utils.constants import COUNTRY_MAPPING_PATH, SILVER_DIM_COUNTRY
from utils.io_helpers import read_csv_from_volume, write_df_to_table

## Read and write

Read the CSV, do a quick visual check, then write it as a Delta table.

In [0]:
from pyspark.sql import functions as F

country_df = read_csv_from_volume(spark, COUNTRY_MAPPING_PATH)

# MDG and MAD both map to Madagascar - keep MAD (the IOC code, which matches what the dataset uses)
country_df = country_df.filter(F.col("country_code") != "MDG")

print(f"Rows in country mapping: {country_df.count()}")
country_df.show(10, truncate=False)

write_df_to_table(country_df, SILVER_DIM_COUNTRY)
print(f"Wrote {SILVER_DIM_COUNTRY}")